In [ ]:
import sys
import os
import torch
import matplotlib.pyplot as plt
import numpy as np
sys.path.append("..")
from models.autoencoder_classifier import *

import pandas as pd
from utils.balanced_builders_hdf import *
import torch.nn.functional as F
from scipy.signal import find_peaks
from tqdm import tqdm
from captum.attr import IntegratedGradients
from captum.attr import Occlusion
from captum.attr import GradientShap
from scipy.signal import find_peaks
from final_pipline.encode_xrd import *
from final_pipline.xrd_sum_cosine import *
from config import *
from Composition.pipelines import *

In [3]:
cs = build_balanced_cs_loaders_from_h5(
    h5_path=xrd_dataset,
    per_class_cs=105000,   
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=4,
)
print(cs["counts"])
print(cs["sizes"])  

{'triclinic': 105000, 'monoclinic': 105000, 'orthorhombic': 105000, 'tetragonal': 105000, 'trigonal': 105000, 'hexagonal': 105000, 'cubic': 105000}
{'train': 588000, 'val': 73500, 'test': 73500}


In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_cls = DeepConvAutoencoderClassifier(
    input_length=cs["input_len"],
    latent_dim=64,
    cls_dim=128,
    num_classes=cs["num_classes"],
    use_projection_head=True
).to(device)
save_path_clf = CS_Cls
state_dict = torch.load(save_path_clf, map_location=device, weights_only=True)
model_cls.load_state_dict(state_dict)
model_cls.eval();


In [20]:
mean_patterns = compute_mean_xrd_patterns_by_class(
    cs["train_loader"],
    class_names=cs["class_names"],
    max_patterns_per_class=200
)

# Stack → (num_classes, input_len)
mean_xrd_np = np.stack([mean_patterns[i] for i in range(cs["num_classes"])])

# Convert to torch
mean_xrd = torch.tensor(mean_xrd_np, dtype=torch.float32).to(device)

# Add channel dimension if needed
# Typical Conv1D input: (B, 1, L)
mean_xrd = mean_xrd.unsqueeze(1)
with torch.no_grad():
    z_rec_means, z_cls_means = model_cls.encode(mean_xrd)
    z_cls_means = torch.nn.functional.normalize(z_cls_means, dim=1)


In [ ]:
class_prototypes = {
    "z_cls_means": z_cls_means.cpu(),                
    "class_names": cs["class_names"],                
    "class_to_idx": {name: i for i, name in enumerate(cs["class_names"])},
    "idx_to_class": {i: name for i, name in enumerate(cs["class_names"])},
}
torch.save(class_prototypes, "xrd_class_prototypes.pth")


In [ ]:
def get_test_samples(test_loader, num_samples=32):
    xs, ys = [], []

    for x, y in test_loader:
        # x must be RAW XRD
        # expected shape: (B, 1, 2048)
        xs.append(x)
        ys.append(y)

        if torch.cat(xs).shape[0] >= num_samples:
            break

    x_test = torch.cat(xs)[:num_samples]
    y_test = torch.cat(ys)[:num_samples]

    return x_test, y_test

In [ ]:
x_test, y_test = get_test_samples(cs["test_loader"], num_samples=32)
x_test = x_test.to(device)

In [ ]:
print("x_test shape:", x_test.shape)
print("x_test dtype:", x_test.dtype)
print("x_test min/max:", x_test.min().item(), x_test.max().item())

x_test shape: torch.Size([32, 2048])
x_test dtype: torch.float32
x_test min/max: 0.0 1.0


In [ ]:
proto = torch.load("xrd_class_prototypes.pth", map_location=device)

z_cls_means = proto["z_cls_means"].to(device)
class_names = proto["class_names"]


with torch.no_grad():
    _, z_cls_test = model_cls.encode(x_test)
    z_cls_test = torch.nn.functional.normalize(z_cls_test, dim=1)

cos_sim = z_cls_test @ z_cls_means.T
pred_idx = cos_sim.argmax(dim=1)
pred_labels = [class_names[i] for i in pred_idx.tolist()]


In [ ]:
for i in range(len(pred_labels)):
    gt = cs["class_names"][y_test[i].item()]
    pred = pred_labels[i]
    score = cos_sim[i, pred_idx[i]].item()

    print(f"{i:02d} | GT={gt:<14} | Pred={pred:<14} | cos={score:.3f}")

00 | GT=hexagonal      | Pred=cubic          | cos=0.690
01 | GT=orthorhombic   | Pred=triclinic      | cos=0.388
02 | GT=hexagonal      | Pred=cubic          | cos=0.624
03 | GT=cubic          | Pred=cubic          | cos=0.609
04 | GT=orthorhombic   | Pred=cubic          | cos=0.777
05 | GT=trigonal       | Pred=cubic          | cos=0.577
06 | GT=orthorhombic   | Pred=cubic          | cos=0.711
07 | GT=hexagonal      | Pred=hexagonal      | cos=0.581
08 | GT=orthorhombic   | Pred=cubic          | cos=0.638
09 | GT=orthorhombic   | Pred=cubic          | cos=0.637
10 | GT=cubic          | Pred=cubic          | cos=0.743
11 | GT=hexagonal      | Pred=cubic          | cos=0.766
12 | GT=trigonal       | Pred=cubic          | cos=0.557
13 | GT=orthorhombic   | Pred=cubic          | cos=0.577
14 | GT=orthorhombic   | Pred=cubic          | cos=0.595
15 | GT=cubic          | Pred=cubic          | cos=0.748
16 | GT=trigonal       | Pred=cubic          | cos=0.688
17 | GT=monoclinic     | Pred=c

In [26]:
y_true = y_test.cpu()
y_pred = pred_idx.cpu()

correct = (y_true == y_pred).sum().item()
total = len(y_true)

proto_acc = correct / total * 100
print(f"Prototype (cosine) Accuracy = {proto_acc:.2f}% ({correct}/{total})")


Prototype (cosine) Accuracy = 15.62% (5/32)
